In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import Adam
from torch.nn import Module, Sequential, Linear, ReLU, Dropout, RNN, LSTM
from torch.nn.functional import mse_loss
from sklearn.preprocessing import MinMaxScaler
from torch import Tensor
from typing import Iterator
Loader = Iterator[Tensor]

In [ ]:
PATH  = "../../data/deep learning project data/NN-data.csv"
GROUP = "tressure group"
VAL   = "volume"
YEAR  = "year"
MONTH = "month"

In [ ]:
PYEAR  = 2024
EPOCHS = 250
BSIZE  = 3
LRATE  = 1e-3
SLEN   = 3

<br>

### Organizing the data

In [ ]:
data = pd.read_csv(PATH, usecols = [GROUP, YEAR, MONTH, VAL])
data.head()

Summing duplicates

In [ ]:
data = data.groupby(data.columns.difference([VAL]).to_list(), as_index = False).sum()

<br>

### Preparing for training

Setting tressure groups as features

In [ ]:
data = data.pivot_table(index = [YEAR, MONTH], columns = GROUP, values = VAL, fill_value = 0).sort_index()

In [ ]:
train = data.loc[data.index.get_level_values(level = YEAR) != PYEAR]
test  = data.loc[data.index.get_level_values(level = YEAR) == PYEAR]

Filtering for full years only

In [ ]:
train = train.groupby(level = YEAR).filter(lambda g : g.count().min() == 12)

Min-Max normalizing each group. Preserves order and sets a [0, 1] range.

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train);

In [ ]:
target = train.loc[train.index.get_level_values(level = MONTH) > 1]
train  = train.loc[train.index.get_level_values(level = MONTH) < 12]

Setting years as batches and months as the sequance.

In [ ]:
def to_batches(table: pd.DataFrame) -> Tensor:
    return torch.stack([torch.FloatTensor(scaler.transform(t)) for year, t in table.groupby(level = YEAR)])

train_ten  = to_batches(train)
target_ten = to_batches(target)
test_ten   = to_batches(test)


In [ ]:
assert train_ten.size(2) == target_ten.size(2) == test_ten.size(2)
assert train_ten.size(1) == target_ten.size(1) == 11
assert train_ten.size(0) == target_ten.size(0)
assert test_ten.size(0) == 1

In [ ]:
tdataset = TensorDataset(train_ten, target_ten)
loader : Loader = DataLoader(tdataset, batch_size = BSIZE, shuffle = True)

<br>

### Building architactures

In [ ]:
class Forecaster(Module):
    def fit(self, loader: Loader, epochs: int, lr: float): pass
    def forecast(self, seed: Tensor, fh: int): pass

In [ ]:
class MyRNN(Forecaster):

    def __init__(self, input_dim: int, lstm_hid: int = 1000, lin_hid: int = 1000, dropr: float = 0.3):
        super().__init__()
        
        self.rnn = LSTM(input_dim, lstm_hid, batch_first = True, dropout = dropr, num_layers = 2)
        self.head = Sequential(Dropout(dropr),
                               Linear(lstm_hid, input_dim))


    def forward(self, x: Tensor, hid = None) -> tuple[Tensor, Tensor]:
        x, hid = self.rnn(x, hid)
        x = self.head(x)
        return x, hid


    def fit(self, loader: Loader, epochs: int, lr: float):
        
        optimizer = Adam(self.parameters(), lr=lr)
        self.train()

        for epoch in range(1, epochs + 1):
            total_loss = 0.0

            for data_batch, target_batch in loader:
                optimizer.zero_grad()
                preds, _ = self.forward(data_batch)

                loss = mse_loss(preds, target_batch)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            if epoch % 10 == 0:
                avg_loss = total_loss / len(loader)
                print(f"Epoch {epoch:3d} | Avg Loss: {avg_loss:.4f}")



    @torch.no_grad
    def forecast(self, seed: Tensor, fh: int, hid = None):
        
        self.eval()

        seed_len = seed.size(1)

        for step in range(seed_len):
            val = seed[:, step, :]
            yield val
            _, hid = self.forward(val, hid)

        for step in range(fh - seed_len):
            val, hid = self.forward(val, hid)
            yield val

<br>

### Training

In [ ]:
model = MyRNN(input_dim = train_ten.size(-1))
model.fit(loader, EPOCHS, LRATE)

<br>

### Forecasting

In [ ]:
pred = torch.concat(tuple(model.forecast(test_ten[:, :SLEN, :], test_ten.size(1))))

In [ ]:
assert pred.shape == test.shape
assert pred.size(0) == test.index.size

In [ ]:
pred = pred.detach().numpy()
test_ten = test_ten.detach().numpy().squeeze()

In [ ]:
pred = scaler.inverse_transform(pred)
test_ten = scaler.inverse_transform(test_ten)

In [ ]:
print(f"total prediction: {pred.sum():.2e}")

In [ ]:
for x, y in zip(pred.T, test_ten.T):

    plt.plot(test.index.get_level_values(level = MONTH), x)
    plt.plot(test.index.get_level_values(level = MONTH), y)
    
    plt.legend(['pred', 'actual'])
    plt.show()